# Replica Movement in Weaviate

Replica Movement in Weaviate (available since v1.32) allows you to actively manage where shard replicas live within your cluster — without recreating collections or taking downtime.

This is useful in scenarios such as:

- Scaling up: After adding new nodes to a cluster, redistribute replicas so the new nodes share the load.
- Scaling down / node decommissioning: Move replicas away from nodes you plan to remove.
- Data locality optimization: Place replicas closer to the workloads that query them most.

## The environment: Multi Node Cluster

In order to work with Replica Moviment, we need a multi node cluster. We use our helm chart. [For more information on how to install it here](https://docs.weaviate.io/deploy/installation-guides/k8s-installation#weaviate-helm-chart)

We will first create a 3 node cluster, using Weaviate's official helm chart

In [ ]:
!pip install weaviate-client tqdm

In [18]:
!helm upgrade --install my-weaviate weaviate/weaviate \
  --namespace my-weaviate \
  --create-namespace \
  --set replicas=3 \
  --set env.REPLICA_MOVEMENT_ENABLED="true"

/opt/homebrew/Cellar/python@3.14/3.14.0/Frameworks/Python.framework/Versions/3.14/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=75860) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Release "my-weaviate" has been upgraded. Happy Helming!
NAME: my-weaviate
LAST DEPLOYED: Mon Apr 13 17:55:38 2026
NAMESPACE: my-weaviate
STATUS: deployed
REVISION: 6
DESCRIPTION: Upgrade complete
TEST SUITE: None


In [ ]:
!kubectl get pods --namespace my-weaviate

Perfect! we now have our cluster available. Let's forward two services (http and grpc) so we can connect from our client. Let's list them first:

In [ ]:
!kubectl get services --namespace my-weaviate

Run the command below **on a new terminal, and keep it running**

In [ ]:
# kubectl port-forward service/weaviate -n my-weaviate 8080:80 & kubectl port-forward service/weaviate-grpc -n my-weaviate 50051:50051

we should now be able to connect to our cluster, list our nodes and shards

In [19]:
import weaviate

client = weaviate.connect_to_local()

print("Client is ready:", client.is_ready())

for node in client.cluster.nodes(output="verbose"):
    print("Node name:", node.name, "with shards count:", len(node.shards))

print("Number of nodes:", len(client.cluster.nodes()))

Client is ready: True
Node name: weaviate-0 with shards count: 0
Node name: weaviate-1 with shards count: 0
Node name: weaviate-2 with shards count: 0
Number of nodes: 3


Now, let's push some data. We want a Multi Tenant collection with 5 Tenants.

In [ ]:
import weaviate

In [20]:

from weaviate.classes.config import Configure

client.collections.delete("MyCollection")

collection = client.collections.create(
    "MyCollection",
    multi_tenancy_config=Configure.multi_tenancy(
        enabled=True,
        auto_tenant_activation=True,
        auto_tenant_creation=True
    ),
    replication_config=Configure.replication(
        factor=3,
        async_enabled=True
    )
)

In [21]:
# lets create a helper function to create objects in batch for a given tenant
# add some tqdm progress bar to it as well
from tqdm import tqdm
def create_objects(client:weaviate.WeaviateClient, collection_name, tenant_id, num_objects):
    collection = client.collections.use(collection_name).with_tenant(tenant_id)
    with collection.batch.fixed_size(100) as batch:
        for i in tqdm(range(num_objects), desc=f"Creating objects for {tenant_id}"):
            obj = {
                "name": f"Object {i} at tenant {tenant_id}",
            }
            batch.add_object(obj, vector=[0.1, 0.2, 0.3])

In [22]:
# let's add 1k objects to Tenant1, 2k To Tenant2 and 3k to Tenant3 and 40k to tenant4
# this can take around 1 minute to complete
create_objects(client, "MyCollection", "Tenant1", 1000)
create_objects(client, "MyCollection", "Tenant2", 2000)
create_objects(client, "MyCollection", "Tenant3", 3000)
create_objects(client, "MyCollection", "Tenant4", 10000)

Creating objects for Tenant1:   0%|          | 0/1000 [00:00<?, ?it/s]

Creating objects for Tenant4: 100%|██████████| 10000/10000 [00:19<00:00, 507.93it/s]


great, now let's check the distribution of the shards across the nodes again, we should see that all the nodes have some shards for tenant1, tenant2, tenant3 and tenant4, but one of the nodes should have more shards for tenant4 as it has more objects than the other tenants

#### Notice: Object count may have a delay.

In [27]:
for node in client.cluster.nodes(output="verbose"):
    print("#" * 5, "Node:", node.name, "with shards count:", len(node.shards))
    for shard in node.shards:
        print("- Shard name:", shard.name, "and objects count:", shard.object_count)
    print("")
    

##### Node: weaviate-0 with shards count: 4
- Shard name: Tenant3 and objects count: 3000
- Shard name: Tenant2 and objects count: 2000
- Shard name: Tenant1 and objects count: 1000
- Shard name: Tenant4 and objects count: 10000

##### Node: weaviate-1 with shards count: 4
- Shard name: Tenant3 and objects count: 3000
- Shard name: Tenant4 and objects count: 10000
- Shard name: Tenant2 and objects count: 2000
- Shard name: Tenant1 and objects count: 1000

##### Node: weaviate-2 with shards count: 4
- Shard name: Tenant4 and objects count: 10000
- Shard name: Tenant1 and objects count: 1000
- Shard name: Tenant3 and objects count: 3000
- Shard name: Tenant2 and objects count: 2000



As you can see, our Tenant4 is way bigger than it's peers. 

Let's move it shards to a new node. So first, let's add some new nodes to our cluster

In [29]:
!helm upgrade --install my-weaviate weaviate/weaviate \
  --namespace my-weaviate \
  --create-namespace \
  --set replicas=6 \
  --set env.REPLICA_MOVEMENT_ENABLED="true"

Release "my-weaviate" has been upgraded. Happy Helming!
NAME: my-weaviate
LAST DEPLOYED: Mon Apr 13 18:00:21 2026
NAMESPACE: my-weaviate
STATUS: deployed
REVISION: 8
DESCRIPTION: Upgrade complete
TEST SUITE: None


Here is a way to check where you Collections and shards are stored:

In [31]:
sharding_state = client.cluster.query_sharding_state(
    collection="MyCollection",
    # shard=shard_name,  # Optional: specify a shard to filter results
)

print(f"Shards in 'MyCollection': {[s.name for s in sharding_state.shards]}")
for shard in sharding_state.shards:
    print(f"Nodes for shard '{shard.name}': {shard.replicas}")

Shards in 'MyCollection': ['Tenant1', 'Tenant2', 'Tenant3', 'Tenant4']
Nodes for shard 'Tenant1': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant2': ['weaviate-2', 'weaviate-0', 'weaviate-1']
Nodes for shard 'Tenant3': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant4': ['weaviate-0', 'weaviate-2', 'weaviate-1']


Great! We now can move our Tenant4 shards to our new nodes and rebalance our cluster
Tenant4 on weaviate-0 goes to weaviate-3
Tenant4 on weaviate-1 goes to weaviate-4
Tenant4 on weaviate-2 goes to weaviate-5


In [ ]:
from weaviate.cluster.models import ReplicationType

t4_replicate = client.cluster.replicate(
    collection="MyCollection",
    shard="Tenant4",
    source_node="weaviate-0",
    target_node="weaviate-3",
    replication_type=ReplicationType.MOVE
)

In [36]:
# now let's see how this operation is:
op_status = client.cluster.replications.get(
    uuid=t4_replicate,
    include_history=True
)
print(f"Status for {t4_replicate}: {op_status.status}")
for history in op_status.status_history:
    print(f"- {history.state}: {history.errors}")

Status for 85bd1172-d50d-4218-8077-c6809f39f0a3: ReplicateOperationStatus(state=<ReplicateOperationState.READY: 'READY'>, errors=[])
- ReplicateOperationState.REGISTERED: []
- ReplicateOperationState.HYDRATING: []
- ReplicateOperationState.FINALIZING: []
- ReplicateOperationState.DEHYDRATING: []


In [33]:
# Let's list all status
all_ops = client.cluster.replications.list_all()
print(f"Total replication operations: {len(all_ops)}")

filtered_ops = client.cluster.replications.query(
    collection="MyCollection",
    target_node="weaviate-3",
)
print(
    f"Filtered operations for collection 'MyCollection' on 'weaviate-3': {len(filtered_ops)}"
)
for op in filtered_ops:
    print("\n", op)

Total replication operations: 1
Filtered operations for collection 'MyCollection' on 'weaviate-3': 1

 _ReplicateOperation(collection='MyCollection', shard='Tenant4', source_node='weaviate-0', status=ReplicateOperationStatus(state=<ReplicateOperationState.FINALIZING: 'FINALIZING'>, errors=[]), status_history=None, target_node='weaviate-3', transfer_type=<ReplicationType.MOVE: 'MOVE'>, uuid=UUID('85bd1172-d50d-4218-8077-c6809f39f0a3'))


In [37]:
# Now let's check if it really moved
sharding_state = client.cluster.query_sharding_state(
    collection="MyCollection",
    # shard=shard_name,  # Optional: specify a shard to filter results
)

print(f"Shards in 'MyCollection': {[s.name for s in sharding_state.shards]}")
for shard in sharding_state.shards:
    print(f"Nodes for shard '{shard.name}': {shard.replicas}")

Shards in 'MyCollection': ['Tenant1', 'Tenant2', 'Tenant3', 'Tenant4']
Nodes for shard 'Tenant1': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant2': ['weaviate-2', 'weaviate-0', 'weaviate-1']
Nodes for shard 'Tenant3': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant4': ['weaviate-2', 'weaviate-1', 'weaviate-3']


In [38]:
# lets move our other shards to their new nodes
client.cluster.replicate(
    collection="MyCollection",
    shard="Tenant4",
    source_node="weaviate-1",
    target_node="weaviate-4",
    replication_type=ReplicationType.MOVE
)

client.cluster.replicate(
    collection="MyCollection",
    shard="Tenant4",
    source_node="weaviate-2",
    target_node="weaviate-5",
    replication_type=ReplicationType.MOVE
)

UUID('ee2bcfe5-f165-48ed-bed5-8bde184daeee')

In [71]:
# Now let's check if it really moved
sharding_state = client.cluster.query_sharding_state(
    collection="MyCollection",
    # shard=shard_name,  # Optional: specify a shard to filter results
)

print(f"Shards in 'MyCollection': {[s.name for s in sharding_state.shards]}")
for shard in sharding_state.shards:
    print(f"Nodes for shard '{shard.name}': {shard.replicas}")

Shards in 'MyCollection': ['Tenant3', 'Tenant4', 'Tenant1', 'Tenant2']
Nodes for shard 'Tenant3': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant4': ['weaviate-3', 'weaviate-4', 'weaviate-5']
Nodes for shard 'Tenant1': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant2': ['weaviate-2', 'weaviate-0', 'weaviate-1']


In [69]:
for node in client.cluster.nodes(output="verbose"):
    print("#" * 5, "Node:", node.name, "with shards count:", len(node.shards))
    for shard in node.shards:
        print("- Shard name:", shard.name, "and objects count:", shard.object_count)
    print("")
    

##### Node: weaviate-0 with shards count: 3
- Shard name: Tenant1 and objects count: 1000
- Shard name: Tenant2 and objects count: 2000
- Shard name: Tenant3 and objects count: 3000

##### Node: weaviate-1 with shards count: 3
- Shard name: Tenant3 and objects count: 3000
- Shard name: Tenant1 and objects count: 1000
- Shard name: Tenant2 and objects count: 2000

##### Node: weaviate-2 with shards count: 4
- Shard name: Tenant2 and objects count: 2000
- Shard name: Tenant4 and objects count: 10000
- Shard name: Tenant1 and objects count: 1000
- Shard name: Tenant3 and objects count: 3000

##### Node: weaviate-3 with shards count: 1
- Shard name: Tenant4 and objects count: 10000

##### Node: weaviate-4 with shards count: 1
- Shard name: Tenant4 and objects count: 10000

##### Node: weaviate-5 with shards count: 1
- Shard name: Tenant4 and objects count: 10000



The same way we have MOVED our shard to different nodes, we can also move them back and remove any empty nodes.

In [82]:
client.cluster.replicate(
    collection="MyCollection",
    shard="Tenant4",
    source_node="weaviate-3",
    target_node="weaviate-0",
    replication_type=ReplicationType.MOVE   
)
client.cluster.replicate(
    collection="MyCollection",
    shard="Tenant4",
    source_node="weaviate-4",
    target_node="weaviate-1",
    replication_type=ReplicationType.MOVE   
)
client.cluster.replicate(
    collection="MyCollection",
    shard="Tenant4",
    source_node="weaviate-5",
    target_node="weaviate-2",
    replication_type=ReplicationType.MOVE   
)

UUID('27625065-bea8-4f8e-a7f5-089e5dbf51ea')

Now let's check if it really moved
### NOTE: This can take some time

In [180]:
sharding_state = client.cluster.query_sharding_state(
    collection="MyCollection",
    #shard="Tenant4",  # Optional: specify a shard to filter results
)

print(f"Shards in 'MyCollection': {[s.name for s in sharding_state.shards]}")
for shard in sharding_state.shards:
    print(f"Nodes for shard '{shard.name}': {shard.replicas}")

Shards in 'MyCollection': ['Tenant4', 'Tenant1', 'Tenant2', 'Tenant3']
Nodes for shard 'Tenant4': ['weaviate-2', 'weaviate-0', 'weaviate-1']
Nodes for shard 'Tenant1': ['weaviate-0', 'weaviate-2', 'weaviate-1']
Nodes for shard 'Tenant2': ['weaviate-2', 'weaviate-0', 'weaviate-1']
Nodes for shard 'Tenant3': ['weaviate-0', 'weaviate-2', 'weaviate-1']


great! Now let's remove our extra nodes.

In [182]:
!helm upgrade --install my-weaviate weaviate/weaviate \
  --namespace my-weaviate \
  --create-namespace \
  --set replicas=3 \
  --set env.REPLICA_MOVEMENT_ENABLED="true"

/opt/homebrew/Cellar/python@3.14/3.14.0/Frameworks/Python.framework/Versions/3.14/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=75860) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Release "my-weaviate" has been upgraded. Happy Helming!
NAME: my-weaviate
LAST DEPLOYED: Mon Apr 13 18:20:33 2026
NAMESPACE: my-weaviate
STATUS: deployed
REVISION: 9
DESCRIPTION: Upgrade complete
TEST SUITE: None


#### NOTE: you may need to forward ports again

by running in terminal

```kubectl port-forward service/weaviate -n my-weaviate 8080:80 & kubectl port-forward service/weaviate-grpc -n my-weaviate 50051:50051```

In [185]:
# lets check our nodes again
for node in client.cluster.nodes(output="verbose"):
    print("Node name:", node.name, "with shards count:", len(node.shards))

print("Number of nodes:", len(client.cluster.nodes()))

Node name: weaviate-0 with shards count: 4
Node name: weaviate-1 with shards count: 4
Node name: weaviate-2 with shards count: 4
Number of nodes: 3


In [188]:
# lets fetch some objects:
collection = client.collections.use("MyCollection").with_tenant("Tenant4")
results = collection.query.fetch_objects(limit=10)
for o in results.objects:
    print(f"Object ID: {o.uuid}, Name: {o.properties['name']}")

Object ID: 000651c7-5654-49b6-b2e4-c895a330b157, Name: Object 2203 at tenant Tenant4
Object ID: 0009df83-9c42-4716-9d04-9d0ccec8a5fa, Name: Object 8805 at tenant Tenant4
Object ID: 000afe5d-f0d3-460d-9b96-74707d71a88b, Name: Object 105 at tenant Tenant4
Object ID: 000d7aff-edc2-44a0-84cb-353b6a366d08, Name: Object 9232 at tenant Tenant4
Object ID: 0012a727-b9d7-49c0-96ff-8afd382abe76, Name: Object 8983 at tenant Tenant4
Object ID: 00206124-c1f1-498a-bf76-3f4551826572, Name: Object 5867 at tenant Tenant4
Object ID: 0026b962-af22-44da-bb2e-e19c0afbb3ca, Name: Object 7613 at tenant Tenant4
Object ID: 002e14b0-530e-4e6b-b0c2-af4f3106cff0, Name: Object 2528 at tenant Tenant4
Object ID: 003cb6dd-95f0-44b7-b299-917cdf671a96, Name: Object 9075 at tenant Tenant4
Object ID: 003cbea1-0ed8-499a-a37f-50bb60b26dcc, Name: Object 5976 at tenant Tenant4
